# Notebook 06: Full Fine-Tuning with Multi-Scale Attention

In this notebook, we perform true end-to-end training (Full Fine-Tuning) using Multi-Scale representations. 
The raw time series is passed through 4 independent Moirai Encoders (Patch sizes: 64, 32, 16, 8) simultaneously.

We compare three approaches:
1. **Baseline (Flatten Multi-Scale):** Mean pooling on each scale's patches, then concatenation.
2. **Sequential Cross-Scale:** Attention flows from coarse (64) to fine (8) in a cascade.
3. **Parallel Cross-Scale:** All scales interact through a single shared attention layer.

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from IPython.display import display

# Import de tes utilitaires
from encoder import MoiraiEncoder
from heads import *
from models import *
from utils import *

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"

# Paramètres d'entraînement
# BATCH_SIZE réduit pour éviter le Out Of Memory (4 Moirai en parallèle !)
BATCH_SIZE = 256
lr_grid = [1e-4]
wd_grid = [0.05]

# Les tailles de séquences (nombre de patchs) par variable pour le dataset LSST
PATCHES_COUNTS = {
    64: 2, 
    32: 3, 
    16: 4, 
    8: 6
}

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Chargement des données
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)
print(f"Data loaded. Number of classes: {num_classes}")

Data loaded. Number of classes: 14


In [3]:
from peft import get_peft_model, LoraConfig

class MultiScaleSharedLoraWrapper(nn.Module):
    """
    Wrapper Multi-Échelles avec UN SEUL encodeur partagé, fine-tuné via DoRA.
    """
    def __init__(self, head_class, head_kwargs, num_vars=6, size="small", remove_last_patch=False, lora_r=8):
        super().__init__()
        self.patch_sizes = [64, 32, 16, 8] 
        self.num_vars = num_vars
        self.remove_last_patch = remove_last_patch
        
        # 1. On charge le Foundation Model UNE SEULE FOIS
        shared_moirai = MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{size}")
        
        # 2. On lui applique DoRA
        target_modules = "all-linear"

        peft_config = AdaLoraConfig(
                target_r=lora_r,                
                init_r=lora_r + 4,              
                lora_alpha=lora_r * 2,
                target_modules=target_modules,
                tinit=200,                      
                tfinal=1000,                    
                deltaT=10,                      
                total_step=3000
            )
        shared_moirai_peft = get_peft_model(shared_moirai, peft_config)
        
        # 3. On instancie 4 encodeurs qui pointent vers ce MÊME module DoRA
        encoders_dict = {}
        for p in self.patch_sizes:
            enc = MoiraiEncoder(
                module=shared_moirai_peft, 
                prediction_length=p,  
                context_length=36, 
                patch_size=p, 
                num_samples=100, 
                target_dim=num_vars, 
                feat_dynamic_real_dim=0, 
                past_feat_dynamic_real_dim=0
            )
            encoders_dict[str(p)] = enc
            
        self.encoders = nn.ModuleDict(encoders_dict)
        
        # 4. Initialisation de la tête
        self.head = head_class(**head_kwargs)

    def forward(self, target, obs, pad):
        features_list = []
        for p in self.patch_sizes:
            # Extraction via le DoRA partagé
            Z = self.encoders[str(p)](target, obs, pad)
            
            if self.remove_last_patch:
                B, S, F = Z.shape
                P = S // self.num_vars
                Z_reshaped = Z.view(B, self.num_vars, P, F)
                Z_no_mask = Z_reshaped[:, :, :-1, :]
                Z = Z_no_mask.reshape(B, -1, F)
                
            features_list.append(Z)
        
        # Concaténation des 4 échelles
        concat_features = torch.cat(features_list, dim=1)
        
        return self.head(concat_features)

In [13]:
BATCH_SIZE = 256
lr_grid = [1e-4]
wd_grid = [0.05]

PATCHES_COUNTS = {64: 2, 32: 3, 16: 4, 8: 6}

tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)

In [7]:
# Définition de toutes les combinaisons à tester
experiences = [
    ("Baseline: Flatten Multi-Scale", MultiScaleMeanPoolingClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS}),
    ("Sequential (Cascade) | Non-Shared", SequentialCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": False}),
    ("Sequential (Cascade) | Shared", SequentialCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": True}),
    ("Parallel (Concat) | Non-Shared", ParallelCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": False}),
    ("Parallel (Concat) | Shared", ParallelCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": True})
]

df_dora_multiscale = pd.DataFrame(index=[exp[0] for exp in experiences], columns=["Test Accuracy"])
df_dora_multiscale.index.name = "Head Approach (Shared DoRA, r=8)"

print("🚀 DÉBUT DE L'ÉTUDE : MULTI-SCALE AVEC DoRA PARTAGÉ\n")

for arch_name, head_class, head_kwargs in experiences:
    print(f"🔄 Test en cours : {arch_name}")
    
    _, test_acc = universal_grid_search(
        model_class=MultiScaleSharedLoraWrapper,
        model_kwargs={
            "head_class": head_class,
            "head_kwargs": head_kwargs,
            "num_vars": NUM_VARS,
            "size": SIZE, 
            "remove_last_patch": False,
            "lora_r": 2
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=wd_grid, lr_grid=lr_grid,
        verbose = True
    )
    
    df_dora_multiscale.loc[arch_name, "Test Accuracy"] = test_acc
    print(test_acc)

print("\n" + "="*50)
print("🏆 RÉSULTATS FINAUX (DoRA Partagé) 🏆")
print("="*50)
display(df_dora_multiscale.astype(float).round(4))

🚀 DÉBUT DE L'ÉTUDE : MULTI-SCALE AVEC DoRA PARTAGÉ

🔄 Test en cours : Baseline: Flatten Multi-Scale
LR=0.0001 | WD=0.05
1.612277213150893
1.4350167328749246
1.3422339514988224
1.2646797352690038
1.2082838411253642
1.164863966344818
1.133967341446295
1.1080788335179894
1.0935698999622003
1.0838430984233454
1.0724388050839184
1.0661327645061462
1.0750838130470213
1.0576324540425122
1.0600536964773162
1.0525462908473442
1.0561003520236751
1.0523776232711668
1.0463908222632679
1.0528860983809805
1.0512004024614163
1.0517269663694429
1.0617657202046091
1.0644469891137225
 Early stopping : 23
Val Loss: 1.0464
Acc on Test Set : 0.6561

0.6561232765612328
🔄 Test en cours : Sequential (Cascade) | Non-Shared
LR=0.0001 | WD=0.05
1.5550083290270673
1.378824328019367
1.2449176747624466
1.1934493537840805
1.1446203342298182
1.1456657502709366
1.1051597081548798
1.1248502120739077
1.118663680262682
1.09378822159961
1.0935625312774162
1.08617457820148
1.1072427896949333
1.1086410206507862
1.1287298309

,Test Accuracy
"Head Approach (Shared DoRA, r=8)",
Baseline: Flatten Multi-Scale,0.6561
Sequential (Cascade) | Non-Shared,0.6517
Sequential (Cascade) | Shared,0.6302
Parallel (Concat) | Non-Shared,0.6448
Parallel (Concat) | Shared,0.6436


In [14]:
# Définition de toutes les combinaisons à tester
experiences = [
    ("Baseline: Flatten Multi-Scale", MultiScaleMeanPoolingClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS}),
    ("Sequential (Cascade) | Non-Shared", SequentialCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": False}),
    ("Sequential (Cascade) | Shared", SequentialCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": True}),
    ("Parallel (Concat) | Non-Shared", ParallelCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": False}),
    ("Parallel (Concat) | Shared", ParallelCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": True})
]

df_dora_multiscale = pd.DataFrame(index=[exp[0] for exp in experiences], columns=["Test Accuracy"])
df_dora_multiscale.index.name = "Head Approach (Shared DoRA, r=8)"

print("🚀 DÉBUT DE L'ÉTUDE : MULTI-SCALE AVEC DoRA PARTAGÉ\n")

for arch_name, head_class, head_kwargs in experiences:
    print(f"🔄 Test en cours : {arch_name}")
    
    _, test_acc = universal_grid_search(
        model_class=MultiScaleSharedWrapper,
        model_kwargs={
            "head_class": head_class,
            "head_kwargs": head_kwargs,
            "num_vars": NUM_VARS,
            "size": SIZE, 
            "remove_last_patch": False,
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=wd_grid, lr_grid=lr_grid,
        verbose=True
    )
    
    df_dora_multiscale.loc[arch_name, "Test Accuracy"] = test_acc
    print(test_acc)

print("\n" + "="*50)
print("🏆 RÉSULTATS FINAUX (DoRA Partagé) 🏆")
print("="*50)
display(df_dora_multiscale.astype(float).round(4))

🚀 DÉBUT DE L'ÉTUDE : MULTI-SCALE AVEC DoRA PARTAGÉ

🔄 Test en cours : Baseline: Flatten Multi-Scale
LR=0.0001 | WD=0.05
1.6641009425729272
1.3623806354476184
1.2204571690985826
1.1369259386527828
1.1102641879058466
1.066453070175357
1.0588594878592141
1.065589320368883
1.0655683443798283
1.1015093966228207
1.1887863680599182
1.1891418646990768
 Early stopping : 11
Val Loss: 1.0589
Acc on Test Set : 0.6557

0.6557177615571776
🔄 Test en cours : Sequential (Cascade) | Non-Shared
LR=0.0001 | WD=0.05
1.5618030627568562
1.2900116463017657
1.1717582117251264
1.124036790878792
1.1145944101054495
1.1049785284492057
1.1325750118348656
1.2111587883011112
1.2849624874146004
1.393070030018566
1.4177811262084217
 Early stopping : 10
Val Loss: 1.1050
Acc on Test Set : 0.6488

0.64882400648824
🔄 Test en cours : Sequential (Cascade) | Shared
LR=0.0001 | WD=0.05
1.5571377267682456
1.2937448538415801
1.1919029505272223
1.1464101551024894
1.1372938960548338
1.1206070126556769
1.1578479133001187
1.26486074

,Test Accuracy
"Head Approach (Shared DoRA, r=8)",
Baseline: Flatten Multi-Scale,0.6557
Sequential (Cascade) | Non-Shared,0.6488
Sequential (Cascade) | Shared,0.6391
Parallel (Concat) | Non-Shared,0.6387
Parallel (Concat) | Shared,0.6257


In [12]:
# Définition de toutes les combinaisons à tester
experiences = [
    ("Baseline: Flatten Multi-Scale", MultiScaleMeanPoolingClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS}),
    ("Sequential (Cascade) | Non-Shared", SequentialCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": False}),
    ("Sequential (Cascade) | Shared", SequentialCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": True}),
    ("Parallel (Concat) | Non-Shared", ParallelCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads":384, "shared_layer": False}),
    ("Parallel (Concat) | Shared", ParallelCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": True})
]

df_dora_multiscale = pd.DataFrame(index=[exp[0] for exp in experiences], columns=["Test Accuracy"])
df_dora_multiscale.index.name = "Head Approach (Shared DoRA, r=8)"

print("🚀 DÉBUT DE L'ÉTUDE : MULTI-SCALE AVEC DoRA PARTAGÉ\n")

for arch_name, head_class, head_kwargs in experiences:
    print(f"🔄 Test en cours : {arch_name}")
    
    _, test_acc = universal_grid_search(
        model_class=MultiScaleFullWrapper,
        model_kwargs={
            "head_class": head_class,
            "head_kwargs": head_kwargs,
            "num_vars": NUM_VARS,
            "size": SIZE, 
            "remove_last_patch": False
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=wd_grid, lr_grid=lr_grid,
        verbose = True
    )
    
    df_dora_multiscale.loc[arch_name, "Test Accuracy"] = test_acc
    print(test_acc)

print("\n" + "="*50)
print("🏆 RÉSULTATS FINAUX (DoRA Partagé) 🏆")
print("="*50)
display(df_dora_multiscale.astype(float).round(4))

🚀 DÉBUT DE L'ÉTUDE : MULTI-SCALE AVEC DoRA PARTAGÉ

🔄 Test en cours : Baseline: Flatten Multi-Scale
LR=1e-05 | WD=0.05
2.2598756348214497
2.0050322854421974
1.8530432925960882
1.7247705498361976
1.6134613684522428
1.5160442600405313
1.4358454729483379
1.371715752089896
1.3215209129380017
1.2777193183821391
1.2443261030243664
1.2145541790055066
1.1901204004520323
1.1691324865914943
1.1535700598383338
1.1376677247566906
1.1286433935165405
1.1111251261176132
1.1097302572513983
1.0969497682602425
1.0903416096679563
1.0828750278891586
1.0818907361689623
1.0789400639572764
1.07509314626213
1.0739767406044938
1.0811030186288726
1.0893344946993075
1.0846732922685824
1.084126559699454
1.090036158639241
 Early stopping : 30
Val Loss: 1.0740
Acc on Test Set : 0.6529

0.6528791565287916
🔄 Test en cours : Sequential (Cascade) | Non-Shared
LR=1e-05 | WD=0.05
2.257238884282306
1.980623264622882


KeyboardInterrupt: 

In [15]:
from peft import get_peft_model, LoraConfig

class MultiScaleNonSharedLoraWrapper(nn.Module):
    """
    Wrapper Multi-Échelles avec 4 encodeurs INDÉPENDANTS, chacun fine-tuné via DoRA.
    """
    def __init__(self, head_class, head_kwargs, num_vars=6, size="small", remove_last_patch=False, lora_r=8):
        super().__init__()
        self.patch_sizes = [64, 32, 16, 8] 
        self.num_vars = num_vars
        self.remove_last_patch = remove_last_patch
        
        target_modules = 'all-linear'
        
        # On instancie 4 encodeurs totalement indépendants
        encoders_dict = {}
        for p in self.patch_sizes:
            # 1. On charge un nouveau Foundation Model pour CHAQUE échelle
            base_moirai = MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{size}")
            
            # 2. Instanciation de l'encodeur
            enc = MoiraiEncoder(
                module=base_moirai, 
                prediction_length=p,  
                context_length=36, 
                patch_size=p, 
                num_samples=100, 
                target_dim=num_vars, 
                feat_dynamic_real_dim=0, 
                past_feat_dynamic_real_dim=0
            )
            
            # 3. On lui applique DoRA individuellement
            peft_config = AdaLoraConfig(
                    target_r=lora_r,                
                    init_r=lora_r + 4,              
                    lora_alpha=lora_r * 2,
                    target_modules=target_modules,
                    tinit=200,                      
                    tfinal=1000,                    
                    deltaT=10,                      
                    total_step=3000
                )
            enc_peft = get_peft_model(enc, peft_config)
            
            encoders_dict[str(p)] = enc_peft
            
        self.encoders = nn.ModuleDict(encoders_dict)
        
        # 4. Initialisation de la tête
        self.head = head_class(**head_kwargs)

    def forward(self, target, obs, pad):
        features_list = []
        for p in self.patch_sizes:
            # Extraction via l'encodeur DoRA spécifique à cette échelle
            Z = self.encoders[str(p)](target, obs, pad)
            
            if self.remove_last_patch:
                B, S, F = Z.shape
                P = S // self.num_vars
                Z_reshaped = Z.view(B, self.num_vars, P, F)
                Z_no_mask = Z_reshaped[:, :, :-1, :]
                Z = Z_no_mask.reshape(B, -1, F)
                
            features_list.append(Z)
        
        # Concaténation des 4 échelles
        concat_features = torch.cat(features_list, dim=1)
        
        return self.head(concat_features)

In [16]:
# Définition de toutes les combinaisons à tester
experiences = [
    ("Baseline: Flatten Multi-Scale", MultiScaleMeanPoolingClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS}),
    ("Sequential (Cascade) | Non-Shared", SequentialCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": False}),
    ("Sequential (Cascade) | Shared", SequentialCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": True}),
    ("Parallel (Concat) | Non-Shared", ParallelCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": False}),
    ("Parallel (Concat) | Shared", ParallelCrossScaleClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patches_counts": PATCHES_COUNTS, "num_heads": 384, "shared_layer": True})
]

df_dora_multiscale = pd.DataFrame(index=[exp[0] for exp in experiences], columns=["Test Accuracy"])
df_dora_multiscale.index.name = "Head Approach (Shared DoRA, r=8)"

print("🚀 DÉBUT DE L'ÉTUDE : MULTI-SCALE AVEC DoRA PARTAGÉ\n")

for arch_name, head_class, head_kwargs in experiences:
    print(f"🔄 Test en cours : {arch_name}")
    
    _, test_acc = universal_grid_search(
        model_class=MultiScaleNonSharedLoraWrapper,
        model_kwargs={
            "head_class": head_class,
            "head_kwargs": head_kwargs,
            "num_vars": NUM_VARS,
            "size": SIZE, 
            "remove_last_patch": False,
            "lora_r": 8
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=wd_grid, lr_grid=lr_grid,
        verbose = True
    )
    
    df_dora_multiscale.loc[arch_name, "Test Accuracy"] = test_acc
    print(test_acc)

print("\n" + "="*50)
print("🏆 RÉSULTATS FINAUX (DoRA Partagé) 🏆")
print("="*50)
display(df_dora_multiscale.astype(float).round(4))

🚀 DÉBUT DE L'ÉTUDE : MULTI-SCALE AVEC DoRA PARTAGÉ

🔄 Test en cours : Baseline: Flatten Multi-Scale
LR=0.0001 | WD=0.05
2.0689399339319245
1.8942335873115352
1.782114316777485
1.6878243646001427
1.6173141971836245
1.558939738971431
1.5095220077328566
1.4687612386253792
1.4337572334258537
1.40333954687041
1.3732480363148014
1.3443255288814142
1.3193347580064603
1.2949523945164874
1.2712813664258011
1.249726288686923
1.2285188475275428
1.2089682391019372
1.1913497292898534
1.174432652752574
1.1606160760895023
1.147381897864303
1.134903436753808
1.122302218181331
1.1136031112050622
1.104026800248681
1.095609237508076
1.0880208907088613
1.0816840165998878
1.0779045597324526
1.0720250974825727
1.065146310542657
1.0613317509007647
1.0576020711805763
1.0543611999449691
1.052175378411766
1.0487051233043516
1.0455403705922568
1.041410073032224
1.0400408758380548
1.0391114552815754
1.0370501745037917
1.0366647679631302
1.0341117091295196
1.0344521757063827
1.0332749519890887
1.0352148718950225
1

,Test Accuracy
"Head Approach (Shared DoRA, r=8)",
Baseline: Flatten Multi-Scale,0.6577
Sequential (Cascade) | Non-Shared,0.6419
Sequential (Cascade) | Shared,0.6302
Parallel (Concat) | Non-Shared,0.6460
Parallel (Concat) | Shared,0.6436
